# World models: predicting transitions

Action-conditioned dynamics, rollouts, and compounding error.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

A world model predicts the next observation or latent state conditioned on actions. Training often uses real previous states; rollouts feed predictions back into the model, allowing errors to accumulate.

This is a controlled linear environment and a tiny neural predictor, not DIAMOND or Genie. Studio’s native world models have training paths; foundation models expose separate inference/training support. Available weights do not imply fine-tuning support. Keep episodes separate across train and validation.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
states = torch.randn(128, 2); actions = torch.randn(128, 1)
next_states = 0.9 * states + actions * torch.tensor([[0.2, -0.1]])
model = nn.Sequential(nn.Linear(3, 24), nn.Tanh(), nn.Linear(24, 2))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.02)
for step in range(100):
    prediction = model(torch.cat([states, actions], -1))
    loss = F.mse_loss(prediction, next_states)
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
real = torch.tensor([[1., -1.]]); imagined = real.clone(); errors = []
with torch.no_grad():
    for step in range(20):
        action = torch.tensor([[0.3]])
        real = 0.9 * real + action * torch.tensor([[0.2, -0.1]])
        imagined = model(torch.cat([imagined, action], -1))
        errors.append(F.mse_loss(imagined, real).item())
print('one-step train MSE:', loss.item())
plt.plot(errors, color='#d73535'); plt.xlabel('rollout step'); plt.ylabel('rollout MSE'); plt.show()

## Try it yourself

Extend the rollout to 100 steps and use actions outside the training range. Does low one-step loss guarantee stable simulation?

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [DIAMOND](https://arxiv.org/abs/2405.12399)
- [DreamerV3](https://arxiv.org/abs/2301.04104)